In [1]:
import numpy as np
import pandas as pd
import polars as pl
import svy
from statsmodels.stats.multitest import multipletests

In [2]:
h5_markers = pd.read_csv('../data/processed/h5_markers.csv')

def enrich(sample_path, weight_col, weight_divisor):
    df = pd.read_csv(sample_path)
    df = pd.merge(df, h5_markers, on=['SEQN', 'cycle'], how='left')
    df[f'{weight_col.lower()}_pooled'] = df[weight_col] / weight_divisor
    return df

In [3]:
# H1 population -> pulse vs triglycerides, plus the pulse-vs-free-T3 calibration check
h1 = enrich('../data/processed/h1_analytic_sample.csv', 'WTSAF2YR', 3)
h1_nobb = h1[~h1['on_beta_blocker']]

design1 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wtsaf2yr_pooled')
sample1 = svy.Sample(pl.from_pandas(h1_nobb), design=design1)

model_pulse_tg = svy.GLM(sample1)
model_pulse_tg.fit(y='LBXTR', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print('=== pulse -> triglycerides ===')
print(model_pulse_tg)

model_calibration = svy.GLM(sample1)
model_calibration.fit(y='BPXPLS', x=['LBXT3F'], family='gaussian')
print()
print('=== calibration: pulse ~ free T3 (descriptive, not part of the formal multiplicity family) ===')
print(model_calibration)

=== pulse -> triglycerides ===
GLM: Gaussian (identity)
  Modeling : LBXTR
  n            : 2967          DF Residuals : 45
  Deviance     : 2.9105e+07    Scale        : 621407.2836
  AIC          : 35705.4523    BIC          : -
  R-squared    : 0.07245       R-sq (adj)   : 0.07119
  F-stat (adj) : 18.11288      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t         P>|t|   [0.025     0.975]   
  ------------  ---------  --------  --------  ------  ---------  ---------
  _intercept_   -41.10782  25.27866  -1.62619  0.1109  -92.02165  9.80600  
  BPXPLS        1.25662    0.30073   4.17853   <0.001  0.65091    1.86232  
  RIDAGEYR      0.62572    0.13456   4.65024   <0.001  0.35471    0.89673  
  RIAGENDR_2.0  -33.16799  5.50043   -6.03007  <0.001  -44.24642  -22.08955
  BMXBMI        2.40583    0.33384   7.20651   <0.001  1.73344    3.07822  

=== calibration: pulse ~ free T3 (descriptive, not part of the formal multiplicity family) ===
GLM: Gaussian (ident

/var/folders/rs/yth10b8d4v1043_hxt3sv4x40000gn/T/ipykernel_9339/2992507828.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{weight_col.lower()}_pooled'] = df[weight_col] / weight_divisor


In [4]:
# H2 population -> pulse and cholesterol vs metabolic syndrome
h2 = enrich('../data/processed/h2_analytic_sample.csv', 'WTSAF2YR', 3)
h2_nobb = h2[~h2['on_beta_blocker']]

design2 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wtsaf2yr_pooled')

sample2_nobb = svy.Sample(pl.from_pandas(h2_nobb), design=design2)
model_pulse_mets = svy.GLM(sample2_nobb)
model_pulse_mets.fit(y='metabolic_syndrome', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='binomial')
print('=== pulse -> metabolic syndrome ===')
print(model_pulse_mets)

sample2_full = svy.Sample(pl.from_pandas(h2), design=design2)
model_chol_mets = svy.GLM(sample2_full)
model_chol_mets.fit(y='metabolic_syndrome', x=['LBXTC', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='binomial')
print()
print('=== cholesterol -> metabolic syndrome ===')
print(model_chol_mets)

=== pulse -> metabolic syndrome ===
GLM: Binomial (logit)
  Modeling : metabolic_syndrome
  n            : 2951          DF Residuals : 45
  Deviance     : 2803.9012     Scale        : 1.0000
  AIC          : 2819.3219     BIC          : -
  R-squared    : 0.24762       R-sq (adj)   : 0.24660
  F-stat (adj) : 68.28962      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t          P>|t|   [0.025     0.975]  
  ------------  ---------  --------  ---------  ------  ---------  --------
  _intercept_   -10.47441  0.76996   -13.60386  <0.001  -12.02519  -8.92364
  BPXPLS        0.02649    0.00606   4.36852    <0.001  0.01428    0.03870 
  RIDAGEYR      0.04854    0.00448   10.83746   <0.001  0.03952    0.05757 
  RIAGENDR_2.0  -0.29368   0.13914   -2.11073   0.0404  -0.57391   -0.01344
  BMXBMI        0.19811    0.01276   15.52120   <0.001  0.17241    0.22382 

=== cholesterol -> metabolic syndrome ===
GLM: Binomial (logit)
  Modeling : metabolic_syndrome
  n      

/var/folders/rs/yth10b8d4v1043_hxt3sv4x40000gn/T/ipykernel_9339/2992507828.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{weight_col.lower()}_pooled'] = df[weight_col] / weight_divisor


In [5]:
# H3 population -> pulse and cholesterol vs log-CRP
h3 = enrich('../data/processed/h3_analytic_sample.csv', 'WTMEC2YR', 2)
h3['log_crp'] = np.log(h3['LBXCRP'])
h3_nobb = h3[~h3['on_beta_blocker']]

design3 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wtmec2yr_pooled')

sample3_nobb = svy.Sample(pl.from_pandas(h3_nobb), design=design3)
model_pulse_crp = svy.GLM(sample3_nobb)
model_pulse_crp.fit(y='log_crp', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print('=== pulse -> log-CRP ===')
print(model_pulse_crp)

sample3_full = svy.Sample(pl.from_pandas(h3), design=design3)
model_chol_crp = svy.GLM(sample3_full)
model_chol_crp.fit(y='log_crp', x=['LBXTC', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print()
print('=== cholesterol -> log-CRP ===')
print(model_chol_crp)

=== pulse -> log-CRP ===
GLM: Gaussian (identity)
  Modeling : log_crp
  n            : 5386          DF Residuals : 28
  Deviance     : 6379.1104     Scale        : 209.1184
  AIC          : 16210.2649    BIC          : -
  R-squared    : 0.27512       R-sq (adj)   : 0.27459
  F-stat (adj) : 462.47507     Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.     Std.Err.  t          P>|t|   [0.025    0.975]  
  ------------  --------  --------  ---------  ------  --------  --------
  _intercept_   -6.08036  0.12512   -48.59554  <0.001  -6.33666  -5.82406
  BPXPLS        0.01633   0.00166   9.82311    <0.001  0.01293   0.01974 
  RIDAGEYR      0.00831   0.00118   7.03347    <0.001  0.00589   0.01072 
  RIAGENDR_2.0  0.25171   0.03715   6.77513    <0.001  0.17561   0.32781 
  BMXBMI        0.08987   0.00299   30.05551   <0.001  0.08374   0.09599 

=== cholesterol -> log-CRP ===
GLM: Gaussian (identity)
  Modeling : log_crp
  n            : 6270          DF Residuals : 28
  Deviance

/var/folders/rs/yth10b8d4v1043_hxt3sv4x40000gn/T/ipykernel_9339/2992507828.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{weight_col.lower()}_pooled'] = df[weight_col] / weight_divisor
/var/folders/rs/yth10b8d4v1043_hxt3sv4x40000gn/T/ipykernel_9339/639521572.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  h3['log_crp'] = np.log(h3['LBXCRP'])


In [ ]:
# Multiple comparison correction for the H5 exploratory family, which is kept separate
# from H1-H4's correction. Mixing a pre-registered confirmatory set with
# a post-hoc exploratory one would muddle what each corrected p-value means.
tests = ['pulse -> triglycerides', 'pulse -> MetS', 'pulse -> log-CRP', 'cholesterol -> MetS', 'cholesterol -> log-CRP']
p_values = [0.0001, 0.0001, 0.0001, 0.0066, 0.0001]

reject_fdr, p_values_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
reject_bonf, p_values_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

for t, p, rf, pf, rb, pb in zip(tests, p_values, reject_fdr, p_values_fdr, reject_bonf, p_values_bonf):
    print(f'{t}: raw p={p:.4f} | FDR-adj p={pf:.4f} (sig={rf}) | Bonferroni-adj p={pb:.4f} (sig={rb})')

pulse -> triglycerides: raw p=0.0001 | FDR-adj p=0.0001 (sig=True) | Bonferroni-adj p=0.0005 (sig=True)
pulse -> MetS: raw p=0.0001 | FDR-adj p=0.0001 (sig=True) | Bonferroni-adj p=0.0005 (sig=True)
pulse -> log-CRP: raw p=0.0001 | FDR-adj p=0.0001 (sig=True) | Bonferroni-adj p=0.0005 (sig=True)
cholesterol -> MetS: raw p=0.0066 | FDR-adj p=0.0066 (sig=True) | Bonferroni-adj p=0.0330 (sig=True)
cholesterol -> log-CRP: raw p=0.0001 | FDR-adj p=0.0001 (sig=True) | Bonferroni-adj p=0.0005 (sig=True)
